In [22]:
import gzip
import os
import math

from functools import partial
from pathlib import Path
from shutil import rmtree
from collections import defaultdict

import einx
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from adam_atan2_pytorch import MuonAdamAtan2
from axial_positional_embedding import ContinuousAxialPositionalEmbedding
from beartype import beartype
from beartype.door import is_bearable
from ema_pytorch import EMA
from rotary_embedding_torch import RotaryEmbedding, apply_rotary_emb
from torch import nn, tensor, cat
from torch.nn import Linear, Module, ModuleList
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam
from torchdiffeq import odeint
from torchvision.utils import save_image
from transfusion_pytorch import Transfusion, print_modality_sample
from einops import rearrange, repeat, reduce, einsum, pack, unpack
from einops.layers.torch import Rearrange
from loguru import logger

# Local/Internal project imports
from vot_utils.data import DATA_DIRECTORY, RESULTS_DIRECTORY
from references.transfusion_pytorch.transfusion_pytorch.transfusion import (
    Transformer,
    ModalityInfo,
    GetPredFlows,
    add_temp_batch_dim,
    append_dims,
    cast_tuple,
    char_tokenize,
    decode_chars,
    default,
    default_to_modality_shape_fn,
    default_modality_length_to_time_fn,
    modality_positions_to_is_modality_mask,
    modality_positions_to_tensor,
    order_modality_positions_by_seq_offset,
    exists,
    get_model_output_to_flow_fn,
    identity,
    pack_one_with_inverse,
    is_tensor,
    is_empty,
    tree_map_tensor,
    apply_fn_modality_type,
    join,
    stack,
    derive_rotary_positions_from_modality_positions,
    eval_decorator,
    create_dataloader,
    LossBreakdown,
    pad_sequence,
    typecheck
)

In [23]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

In [35]:
from tqdm import tqdm

from references.transfusion_pytorch.transfusion_pytorch.transfusion import (
    Attention,
    FeedForward,
    concat_contiguous_text,
    get_tokens_since_rightmost_id,
    min_p_filter,
)


class Transfusion(nn.Module):

    def __init__(
        self,
        num_text_tokens,
        transformer,
        dim_latent=None,
        model_output_clean=True,
        channel_first_latent=False,
        modality_default_shape=None,
        modality_encoder=None,
        modality_decoder=None,
        add_pos_emb=False,
        modality_num_dim=None,
        velocity_consistency_loss_weight=0.1,
        reconstruction_loss_weight=0,
        to_modality_shape_fn=default_to_modality_shape_fn,
        fallback_to_default_shape_if_invalid=False,
        modality_encoder_decoder_requires_batch_dim=True,
        pre_post_transformer_enc_dec=None,
        ignore_index=-1,
        flow_loss_weight=1.0,
        text_loss_weight=1.0,
        odeint_kwargs=dict(atol=1e-5, rtol=1e-5, method="midpoint"),
        eps=1e-2,
        prob_uncond=0.1,
    ):
        super().__init__()

        # transformer

        if isinstance(transformer, dict):
            transformer = Transformer(**transformer)

        self.transformer = transformer
        dim = transformer.dim

        self.dim = dim

        # latent and model dimension not the same
        # make it work for 1 modality for now

        dim_latent = default(dim_latent, dim)

        self.dim_latents = cast_tuple(dim_latent)

        # number of modalities

        self.num_modalities = len(self.dim_latents)

        # whether the latents are accepted to be channel first or channel last
        # if channel first, will be rearrange(c ... -> ... c -> (...) c)

        self.channel_first_latent = cast_tuple(
            channel_first_latent, self.num_modalities
        )
        assert len(self.channel_first_latent) == self.num_modalities

        # functions for converting the sampled language model meta string back to modality shape of tuple[int, ...]

        self.to_modality_shape_fn = cast_tuple(
            to_modality_shape_fn, self.num_modalities
        )

        # default token lengths for respective modality
        # fallback if the language model does not come up with valid dimensions

        if not exists(modality_default_shape) or is_bearable(
            modality_default_shape, tuple[int, ...]
        ):
            modality_default_shape = (modality_default_shape,) * self.num_modalities

        self.modality_default_shape = modality_default_shape

        assert len(self.modality_default_shape) == self.num_modalities

        self.fallback_to_default_shape_if_invalid = fallback_to_default_shape_if_invalid

        # default `modality_num_dim` to `len(modality_default_shape)` if latter is specified but former not

        modality_num_dim = default(
            modality_num_dim, tuple(len(shape) for shape in self.modality_default_shape)
        )

        # specifying the number of dimensions for the modality, which will be hard validated

        self.modality_num_dim = cast_tuple(modality_num_dim, self.num_modalities)

        assert len(self.modality_num_dim) == self.num_modalities

        assert all(
            [
                not exists(ndim) or not exists(shape) or len(shape) == ndim
                for ndim, shape in zip(
                    self.modality_num_dim, self.modality_default_shape
                )
            ]
        )

        # whether to add an extra axial positional embedding per modality

        self.add_pos_emb = cast_tuple(add_pos_emb, self.num_modalities)
        assert len(self.add_pos_emb) == self.num_modalities

        self.pos_emb_mlp = ModuleList([])

        for modality_add_pos_emb, modality_ndim in zip(
            self.add_pos_emb, self.modality_num_dim
        ):

            if not modality_add_pos_emb:
                self.pos_emb_mlp.append(None)
                continue

            assert exists(
                modality_ndim
            ), "`modality_num_dim` must be set if you wish to automatically inject axial positional embeddings"

            pos_generating_mlp = ContinuousAxialPositionalEmbedding(
                dim=dim,
                num_axial_dims=modality_ndim,
            )

            self.pos_emb_mlp.append(pos_generating_mlp)

        # modality encoders and decoders

        modality_encoder = cast_tuple(
            modality_encoder, 1 if exists(modality_encoder) else self.num_modalities
        )
        modality_decoder = cast_tuple(
            modality_decoder, 1 if exists(modality_decoder) else self.num_modalities
        )

        self.modality_encoder = ModuleList(modality_encoder)
        self.modality_decoder = ModuleList(modality_decoder)

        assert len(self.modality_encoder) == self.num_modalities
        assert len(self.modality_decoder) == self.num_modalities

        # auto handle batch dimension for modality encoder / decoder

        self.maybe_add_temp_batch_dim = (
            add_temp_batch_dim
            if modality_encoder_decoder_requires_batch_dim
            else identity
        )

        # store number of text tokens

        self.num_text_tokens = num_text_tokens

        # entire "sentence" start and end id

        num_text_special_ids = 2

        self.sos_id, self.eos_id = num_text_tokens, (num_text_tokens + 1)

        # null text id
        self.null_text_id = num_text_tokens + 2
        num_text_special_ids += 1

        # modality meta, start and end tokens - termed [mom] [som] [eom] in this repo

        num_modality_special_ids = self.num_modalities * 2
        som_eom_tensor = (
            torch.arange(num_modality_special_ids)
            + num_text_tokens
            + num_text_special_ids
        )  # shift to the very end
        som_eom_tensor = rearrange(
            som_eom_tensor, "(start_end m) -> start_end m", start_end=2
        )

        # modality meta, start and end ids

        self.som_ids, self.eom_ids = som_eom_tensor.tolist()

        # char tokenizing for modality meta information

        meta_token_offset = (
            num_text_tokens + num_text_special_ids + num_modality_special_ids
        )

        self.meta_id = meta_token_offset

        self.char_tokenizer = partial(char_tokenize, offset=meta_token_offset + 1)
        self.decode_chars = partial(decode_chars, offset=meta_token_offset + 1)

        num_meta_tokens = 128 + 1

        # prepare pre-post transformer encoder / decoder, for the learnable unets as in paper

        if is_bearable(pre_post_transformer_enc_dec, tuple[Module, Module]):
            pre_post_transformer_enc_dec = (pre_post_transformer_enc_dec,)

        pre_post_transformer_enc_dec = cast_tuple(
            pre_post_transformer_enc_dec, self.num_modalities
        )
        assert len(pre_post_transformer_enc_dec) == self.num_modalities

        # latent to model and back
        # by default will be Linear, with or without rearranges depending on channel_first_latent setting
        # can also be overridden for the unet down/up as in the paper with `pre_post_transformer_enc_dec: tuple[Module, Module]`

        latent_to_model_projs = []
        model_to_latent_projs = []

        for (
            dim_latent,
            one_channel_first_latent,
            enc_dec,
        ) in zip(
            self.dim_latents, self.channel_first_latent, pre_post_transformer_enc_dec
        ):

            pre_attend_enc, post_attend_dec = default(enc_dec, (None, None))

            latent_to_model_proj = (
                Linear(dim_latent, dim) if dim_latent != dim else nn.Identity()
            )
            model_to_latent_proj = Linear(dim, dim_latent, bias=False)

            if one_channel_first_latent:
                latent_to_model_proj = nn.Sequential(
                    Rearrange("b d ... -> b ... d"), latent_to_model_proj
                )
                model_to_latent_proj = nn.Sequential(
                    model_to_latent_proj, Rearrange("b ... d -> b d ...")
                )

                if exists(pre_attend_enc):
                    pre_attend_enc = nn.Sequential(
                        pre_attend_enc, Rearrange("b d ... -> b ... d")
                    )

                if exists(post_attend_dec):
                    post_attend_dec = nn.Sequential(
                        Rearrange("b ... d -> b d ..."), post_attend_dec
                    )

            latent_to_model_projs.append(default(pre_attend_enc, latent_to_model_proj))
            model_to_latent_projs.append(default(post_attend_dec, model_to_latent_proj))

        self.latent_to_model_projs = ModuleList(latent_to_model_projs)
        self.model_to_latent_projs = ModuleList(model_to_latent_projs)

        # relative positions

        self.rotary_emb = RotaryEmbedding(transformer.dim_head)

        # embeddings and un-embeddings

        effective_num_text_tokens = (
            num_text_tokens
            + num_text_special_ids
            + num_modality_special_ids
            + num_meta_tokens
        )

        self.text_embed = nn.Embedding(effective_num_text_tokens, dim)

        self.to_text_logits = Linear(dim, effective_num_text_tokens, bias=False)

        text_only_mask = torch.arange(effective_num_text_tokens) < num_text_tokens
        self.register_buffer("text_only_logits_mask", text_only_mask, persistent=False)

        # loss related

        self.ignore_index = ignore_index
        self.flow_loss_weight = flow_loss_weight
        self.text_loss_weight = text_loss_weight

        # velocity consistency weight - only added if EMA model is passed in during training

        self.velocity_consistency_loss_weight = velocity_consistency_loss_weight

        # additional reconstruction loss, through the decoder

        self.has_recon_loss = reconstruction_loss_weight > 0.0
        self.reconstruction_loss_weight = reconstruction_loss_weight

        # whether model is outputting clean

        self.model_output_clean = model_output_clean
        self.eps = eps

        # flow sampling related

        self.odeint_fn = partial(odeint, **odeint_kwargs)

        self.prob_uncond = prob_uncond

        # dummy loss

        self.register_buffer("zero", tensor(0.0), persistent=False)

    @property
    def device(self):
        return next(self.parameters()).device

    def create_ema(self, beta=0.99, *ema_kwargs) -> EMA:

        ema = EMA(self, beta=beta, forward_method_names=("sample",))

        return ema

    def muon_parameters(self):
        params = []

        for m in self.modules():
            if isinstance(m, Attention):
                params.extend(
                    [
                        *m.to_v.parameters(),
                        *m.to_out.parameters(),
                    ]
                )
            elif isinstance(m, FeedForward):
                params.extend([m.net[0].weight, m.net[-1].weight])

        return params

    @property
    def device(self):
        return next(self.parameters()).device

    def get_modality_info(self, modality_type):
        modality_type = default(modality_type, 0)
        return ModalityInfo(
            encoder=self.modality_encoder[modality_type],
            decoder=self.modality_decoder[modality_type],
            latent_to_model=self.latent_to_model_projs[modality_type].to(device),
            model_to_latent=self.model_to_latent_projs[modality_type].to(device),
            add_pos_emb=self.add_pos_emb[modality_type],
            pos_emb_mlp=self.pos_emb_mlp[modality_type].to(device),
            num_dim=self.modality_num_dim[modality_type],
            dim_latent=self.dim_latents[modality_type],
            default_shape=self.modality_default_shape[modality_type],
            som_id=self.som_ids[modality_type],
            eom_id=self.eom_ids[modality_type],
            to_shape_fn=self.to_modality_shape_fn[modality_type],
            channel_first_latent=self.channel_first_latent[modality_type],
            modality_type=modality_type,
        )

    def get_all_modality_info(self) -> list[ModalityInfo]:
        return [self.get_modality_info(i) for i in range(self.num_modalities)]
        

    def create_dataloader(self, *args, **kwargs):
        return create_dataloader(*args, **kwargs)
    
    @torch.no_grad()
    @eval_decorator
    @typecheck
    def sample(
        self,
        prompt=None,
        max_length=2048,
        text_temperature=1.5,
        text_min_p=0.1,
        cache_kv=False,
        fixed_modality_shape=None,
        init_modality_noise = None,
        modality_steps=16,
        return_unprocessed_modalities=False,
        cfg_scale=3.0,
    ):

        device = self.device

        # take care of prompt being a raw tensor, either text or raw modality (image, video, actions, latents, etc)

        if (
            is_tensor(prompt) and prompt.dtype == torch.float
        ):  # is modality with type 0 implicit
            prompt = (0, prompt)

        prompt_is_modality = isinstance(prompt, tuple)

        if is_tensor(prompt) and prompt.dtype in (
            torch.int,
            torch.long,
        ):  # is text only prompt
            prompt = [prompt]

        elif prompt_is_modality:
            modality_type, modality = prompt

            mod = self.get_modality_info(modality_type)

            if exists(mod.encoder):
                with torch.no_grad():
                    mod.encoder.eval()
                    modality = self.maybe_add_temp_batch_dim(mod.encoder)(
                        modality
                    ).detach()

            modality_shape_tuple = self.get_modality_shape(modality, modality_type)
            modality_shape_str = join([*map(str, modality_shape_tuple)], ",")
            modality_meta_info = self.char_tokenizer(modality_shape_str, device=device)

            prompt = [
                tensor([self.meta_id]),
                modality_meta_info,
                tensor([mod.som_id]),
                (modality_type, modality),
                tensor([mod.eom_id]),
            ]

        # sos

        init_text_seq = tensor([self.sos_id], device=device)

        # just take care of prompt being zero dimensions

        modality_sample = [init_text_seq, *default(prompt, [])]

        # take care of moving to device

        modality_sample = tree_map_tensor(modality_sample, lambda t: t.to(device))
        modality_sample = tree_map_tensor(
            modality_sample, lambda t: rearrange(t, "-> 1") if t.ndim == 0 else t
        )

        modality_sample = concat_contiguous_text(modality_sample)

        *_, last_modality_sample = modality_sample

        curr_length = 0
        curr_modality_id = None
        modality_shape = None

        num_past_modalities = int(
            prompt_is_modality
        )  # either 0 or 1 (if the prompt given is a modality)

        text_is_greedy = text_temperature == 0.0
        is_decoding_text = True  # starts off with text decoding, and alternates with modalities depending on [som] tokens detected

        #TODO: Just use the default shape from the beginning
        # All this function does is set the modality shape and sets decoding text to False
        def maybe_transition_to_modality_decoding(seq):
            nonlocal modality_shape
            nonlocal is_decoding_text
            nonlocal curr_modality_id

            sampled_token_id = seq[-1]

            if sampled_token_id not in self.som_ids:
                return

            curr_modality_id = self.som_ids.index(sampled_token_id)

            if exists(fixed_modality_shape):
                modality_shape = fixed_modality_shape

            # get the tokens after the modality meta id

            maybe_meta_tensor = get_tokens_since_rightmost_id(seq, self.meta_id)

            mod = self.get_modality_info(curr_modality_id)

            default_shape = mod.default_shape
            maybe_modality_num_dim = mod.num_dim
            meta_str_to_modality_shape = mod.to_shape_fn

            if maybe_meta_tensor.numel() > 0:
                meta_tensor = maybe_meta_tensor[:-1]
                meta_str = self.decode_chars(meta_tensor)

                if not meta_str.isdigit() or int(meta_str) <= 0:

                    assert exists(
                        default_shape
                    ), "invalid modality meta information detected, please set `modality_default_shape` in order to properly fallback"
                    modality_shape = default_shape
                else:
                    modality_shape = meta_str_to_modality_shape(meta_str)

            modality_shape = default(modality_shape, default_shape)

            if self.fallback_to_default_shape_if_invalid:

                if (
                    exists(maybe_modality_num_dim)
                    and len(modality_shape) != maybe_modality_num_dim
                ):
                    logger.warning(
                        f"invalid modality shape {modality_shape} for modality {curr_modality_id}. falling back to default shape {default_shape}"
                    )
                    modality_shape = default_shape

            assert exists(
                modality_shape
            ), f"language model did not produce a proper modality shape for modality type {curr_modality_id} - please set a fallback shape with `modality_default_shape`"
            assert not exists(maybe_modality_num_dim) or maybe_modality_num_dim == len(
                modality_shape
            ), f"expected modality type {curr_modality_id} to have {maybe_modality_num_dim} dimensions but language model produced a shape of {modality_shape}"

            is_decoding_text = False

        # determine if to transition from start

        maybe_transition_to_modality_decoding(last_modality_sample)

        cache = None

        with tqdm(total=max_length) as pbar:

            while curr_length <= max_length:

                if is_decoding_text:
                    pbar.set_description("decoding text")

                    *_, seq = modality_sample

                    logits, new_kv_cache = self.forward(
                        [modality_sample],
                        return_loss=False,
                        cache=cache,
                        decode_length=1,
                        decoding_text_or_modality="text",
                        return_kv_cache=True,
                    )

                    logits = logits[0][-1]

                    if text_is_greedy:
                        sampled = logits.argmax(dim=-1, keepdim=True)
                    else:
                        logits = min_p_filter(logits, min_p=text_min_p)

                        probs = (logits / text_temperature).softmax(dim=-1)
                        sampled = torch.multinomial(probs, 1)

                    seq = torch.cat((seq, sampled), dim=-1)
                    modality_sample[-1] = seq

                    pbar.update(1)
                    curr_length += 1

                    if cache_kv:
                        cache = new_kv_cache

                    sampled_token_id = sampled.item()

                    if sampled_token_id == self.eos_id:
                        logger.info(
                            f"detecting an end of string token [{self.eos_id}], terminating sampling early"
                        )
                        break

                    maybe_transition_to_modality_decoding(seq)

                else:
                    assert exists(curr_modality_id)
                    pbar.set_description(f"decoding modality [{curr_modality_id}]")

                    mod = self.get_modality_info(curr_modality_id)

                    modality_length = math.prod(modality_shape)

                    if exists(init_modality_noise):
                        noise = init_modality_noise[:modality_length, : mod.dim_latent]
                    else:
                        assert exists(modality_length)
                        noise = torch.randn(
                            (modality_length, mod.dim_latent), device=device
                        )

                    assert noise.shape == (modality_length, mod.dim_latent)

                    noise = noise.reshape(*modality_shape, mod.dim_latent)

                    if mod.channel_first_latent:
                        noise = rearrange(noise, "... d -> d ...")

                    new_kv_cache = None

                    use_cfg = cfg_scale != 1

                    if use_cfg:
                        # prepare unconditional kv cache for CFG
                        uncond_history = []

                        for item in modality_sample:
                            if is_tensor(item) and item.dtype in (
                                torch.int,
                                torch.long,
                            ):
                                null_tokens = torch.full(
                                    item.shape,
                                    self.null_text_id,
                                    dtype=item.dtype,
                                    device=device,
                                )
                                uncond_history.append(null_tokens)
                            else:
                                uncond_history.append(item)

                        with torch.no_grad():
                            _, uncond_cache = self.forward(
                                [uncond_history],
                                return_loss=False,
                                return_kv_cache=True,
                                return_embed=True,
                                decoding_text_or_modality="modality",
                            )

                    def ode_step_fn(step_times, denoised):
                        nonlocal new_kv_cache

                        # Conditional Input (Text + Image)
                        cond_input = [[*modality_sample, (curr_modality_id, denoised)]]

                        step_times = rearrange(step_times, " -> 1 1")  # batch size of 1
                        step_times = F.pad(
                            step_times, (num_past_modalities, 0), value=1.0
                        )  # past decoded modalities receive a time conditioning of 1.

                        (embeds_cond, get_pred_flows_cond), new_kv_cache = self.forward(
                            cond_input,
                            times=step_times,
                            return_embed=True,
                            cache=cache,
                            decode_length=modality_length,
                            return_kv_cache=True,
                            decoding_text_or_modality="modality",
                        )

                        parse_cond = get_pred_flows_cond[curr_modality_id][-1]
                        parsed_cond = parse_cond(
                            embeds_cond, need_splice=not exists(cache)
                        )
                        cond_flow = add_temp_batch_dim(mod.model_to_latent)(parsed_cond)

                        if not use_cfg:
                            return cond_flow

                        uncond_input = [[*uncond_history, (curr_modality_id, denoised)]]

                        # Unconditional Forward
                        (embeds_uncond, get_pred_flows_uncond), _ = self.forward(
                            uncond_input,
                            times=step_times,  # Same time
                            return_embed=True,
                            cache=uncond_cache,
                            decode_length=modality_length,
                            return_kv_cache=True,
                            decoding_text_or_modality="modality",
                        )

                        parse_uncond = get_pred_flows_uncond[curr_modality_id][-1]
                        parsed_uncond = parse_uncond(embeds_uncond, need_splice=True)
                        uncond_flow = add_temp_batch_dim(mod.model_to_latent)(
                            parsed_uncond
                        )

                        final_flow = uncond_flow + cfg_scale * (cond_flow - uncond_flow)

                        return final_flow

                    times = torch.linspace(0, 1, modality_steps, device=device)

                    trajectory = self.odeint_fn(ode_step_fn, noise, times)

                    # add the sampled modality tokens

                    sampled_modality = trajectory[-1]

                    modality_sample.append((curr_modality_id, sampled_modality))

                    # add the appropriate [eom]

                    eom_id = mod.eom_id
                    modality_sample.append(tensor([eom_id], device=device))

                    # set kv cache if needed

                    if cache_kv:
                        cache = new_kv_cache

                    # back to decoding text

                    pbar.update(modality_length)
                    curr_length += modality_length

                    num_past_modalities += 1
                    curr_modality_id = None
                    modality_length = None

                    is_decoding_text = True

        logger.info(f"sampling stopped at length: {curr_length} / {max_length}")

        if return_unprocessed_modalities:
            return modality_sample

        # post process modality sample, decoding modality types if they have a decoder

        for mod in self.get_all_modality_info():
            decoder_fn = default(mod.decoder, nn.Identity())

            with torch.no_grad():
                decoder_fn.eval()
                modality_sample = apply_fn_modality_type(
                    decoder_fn, modality_sample, modality_type=mod.modality_type
                )

        return modality_sample

    @typecheck
    def forward(
        self,
        modalities,
        times=None,
        num_modalities_to_times_fn=None,
        modality_type=None,
        cache=None,
        decode_length=None,
        decoding_text_or_modality=None,
        velocity_consistency_ema_model=None,
        velocity_consistency_delta_time=1e-3,
        return_only_pred_flows=False,
        return_loss=True,
        return_breakdown=False,
        return_embed=False,
        return_hiddens=False,
        return_kv_cache=False,
        return_times=False,
        prob_uncond=None,
    ):

        is_decoding = exists(decoding_text_or_modality)

        # handle ema model being passed in for velocity consistency loss

        if isinstance(velocity_consistency_ema_model, EMA):
            assert isinstance(velocity_consistency_ema_model.ema_model, Transfusion)
            velocity_consistency_ema_model = velocity_consistency_ema_model.ema_model

        need_velocity_matching = not is_decoding and exists(
            velocity_consistency_ema_model
        )

        # return loss

        return_loss &= not (return_embed or is_decoding)

        batch = len(modalities)
        device = self.device
        tensor_ = partial(tensor, device=device)

        # save a copy for ema model for velocity matching
        velocity_modalities = modalities

        if need_velocity_matching:
            if isinstance(velocity_modalities, list):
                velocity_modalities = [
                    modality.copy() for modality in velocity_modalities
                ]

        # defensively shallow copy out inner lists to prevent in-place mutation of user input
        if isinstance(modalities, list):
            modalities = [
                list(batch) if isinstance(batch, list) else batch
                for batch in modalities
            ]

        # add "sentence" start and end tokens when training

        if return_loss or need_velocity_matching:
            if isinstance(modalities, list):
                for i, modality in enumerate(modalities):
                    modalities[i] = [
                        tensor_([self.sos_id]),
                        *modality,
                        tensor_([self.eos_id]),
                    ]

        # Classifier-free guidance
        prob_uncond = default(prob_uncond, self.prob_uncond)
        if self.training and prob_uncond > 0:
            if isinstance(modalities, list):
                batch = len(modalities)
                rand_mask = torch.rand(batch, device=self.device) < prob_uncond

                new_modalities = []
                for idx, batch_sample in enumerate(modalities):
                    if rand_mask[idx]:
                        # Create unconditional version

                        uncond_sample = []

                        for item in batch_sample:
                            if is_tensor(item) and item.dtype in (
                                torch.int,
                                torch.long,
                            ):
                                null_tokens = torch.full(
                                    item.shape,
                                    self.null_text_id,
                                    dtype=item.dtype,
                                    device=self.device,
                                )
                                uncond_sample.append(null_tokens)
                            else:
                                uncond_sample.append(item)

                        new_modalities.append(uncond_sample)
                    else:
                        new_modalities.append(batch_sample)

                modalities = new_modalities

        # need axial pos emb

        need_axial_pos_emb = any(self.add_pos_emb)

        # standardize modalities to be tuple - type 0 modality is implicit if not given
        # also store modality lengths for determining noising times

        num_modalities = []

        for batch_modalities in modalities:
            batch_num_modalities = 0

            for ind, modality in enumerate(batch_modalities):

                if is_tensor(modality) and modality.dtype == torch.float:
                    modality = (0, modality)

                if not isinstance(modality, tuple):
                    continue

                modality_type, modality_tensor = modality
                batch_modalities[ind] = modality
                batch_num_modalities += 1

            num_modalities.append(batch_num_modalities)

        num_modalities = tensor_(num_modalities)

        # determine the times

        if not exists(times):
            if is_empty(num_modalities) or num_modalities.amax().item() == 0:
                times = torch.empty((batch, 0), device=device, dtype=torch.float)
            else:
                num_modalities_to_times_fn = default(
                    num_modalities_to_times_fn, default_modality_length_to_time_fn
                )

                if exists(num_modalities_to_times_fn):
                    times = num_modalities_to_times_fn(num_modalities)

        # if needs velocity matching, make sure times are in the range of 0 - (1. - <velocity consistency delta time>)

        if need_velocity_matching:
            orig_times = times.clone()
            times = times * (1.0 - velocity_consistency_delta_time)

        # process list of text and modalities interspersed with one another

        modality_positions = []
        modality_tokens = []
        modality_pos_emb = []

        text = []

        modalities = tree_map_tensor(modalities, lambda t: t.to(device))

        # for all modalities, batch process same shaped modalities of the same type

        if not is_decoding:
            for mod in self.get_all_modality_info():
                encode_fn = default(mod.encoder, nn.Identity())

                with torch.no_grad():
                    encode_fn.eval()
                    modalities = apply_fn_modality_type(
                        encode_fn, modalities, modality_type=mod.modality_type
                    )

        # for parsing out the predicted flow from flattened sequence of tokens coming out of transformer

        flows = defaultdict(list)  # store flows for loss

        get_pred_flows: GetPredFlows = defaultdict(
            list
        )  # functions for parsing modalities from Float['b n d'] for model back to latents or pixel space

        def model_to_pred_flow(batch_index, start_index, modality_length, unpack_fn):

            def inner(embed, need_splice=True):
                embed = embed[batch_index]

                if need_splice:
                    if embed.shape[0] < (start_index + modality_length):
                        embed = embed[-modality_length:]
                    else:
                        embed = embed[start_index : (start_index + modality_length)]

                embed = unpack_fn(embed)
                return embed

            return inner

        # for going from predicted flow -> reconstruction

        get_recon_losses = defaultdict(list)

        def get_recon_loss(noise, times, modality):

            def inner(pred_flow):
                recon_modality = noise + pred_flow * (1.0 - times)
                return F.mse_loss(modality, recon_modality)

            return inner

        # prepare storing of sizes of all modalities that require axial positions, for delayed application for efficiency

        pos_emb_max_axial_dims = defaultdict(list)

        # go through all modality samples and do necessary transform

        for batch_index, batch_modalities in enumerate(modalities):

            modality_index = 0
            batch_modality_positions = []
            batch_modality_tokens = []
            batch_modality_pos_emb = []

            batch_text = []

            offset = 0

            for modality in batch_modalities:
                # if non-text modality detected and not given as a tuple
                # cast to (int, Tensor) where int is defaulted to type 0 (convenience for one modality)

                is_text = not isinstance(modality, tuple)
                is_modality = not is_text

                if is_text:
                    modality_tensor = modality
                else:
                    modality_type, modality_tensor, *_ = modality

                # auto move modality tensor to correct device

                mod = self.get_modality_info(modality_type)

                if is_modality:
                    assert (
                        0 <= modality_type < self.num_modalities
                    ), f"received a modality index that is out of range. only {self.num_modalities} modalities specified"

                    channel_dim = 0 if mod.channel_first_latent else -1

                    assert (
                        mod.dim_latent == modality_tensor.shape[channel_dim]
                    ), f"mismatch for modality latent dimension - expected {mod.dim_latent} but received {modality_tensor.shape[-1]} - modality shape is {tuple(modality_tensor.shape)}, perhaps you need to set `channel_first_latent` to the correct value"
                    assert mod.num_dim == (
                        len(modality_tensor.shape) - 1
                    ), f"mismatch for modality number of dimensions - expected {mod.num_dim} but received {len(modality_tensor.shape) - 1} {modality_tensor.shape}"

                # auto ward against scalars (lone start end tokens)

                if (
                    modality_tensor.dtype in (torch.int, torch.long)
                    and modality_tensor.ndim == 0
                ):
                    modality_tensor = rearrange(modality_tensor, "-> 1")

                # handle text

                if is_text:
                    assert modality_tensor.ndim == 1 and modality_tensor.dtype in (
                        torch.int,
                        torch.long,
                    )
                    text_length = modality_tensor.shape[0]

                    batch_text.append(modality_tensor)
                    zeros = torch.zeros(text_length, self.dim, device=device)

                    batch_modality_tokens.append(zeros)

                    offset += text_length

                    if need_axial_pos_emb:
                        batch_modality_pos_emb.append(zeros)

                    continue

                # otherwise handle a modality

                # get times for noising the modality

                modality_time = times[batch_index, modality_index]

                # noise

                if return_loss:
                    noise = torch.randn_like(modality_tensor)

                    noised_modality = modality_tensor * modality_time + noise * (
                        1.0 - modality_time
                    )

                    # the flow is the (data - noise)

                    modality_flow = modality_tensor - noise

                    # append to flow for loss

                    flows[modality_type].append(modality_flow)

                    modality_tensor = noised_modality

                    # store function for deriving reconstruction loss from decoder

                    get_recon_losses[modality_type].append(
                        get_recon_loss(noise, modality_time, modality_tensor)
                    )

                # go through maybe encoder

                modality_tensor = add_temp_batch_dim(mod.latent_to_model)(
                    modality_tensor
                )

                # gather the modality length

                modality_shape_tuple = modality_tensor.shape[:-1]
                modality_length = math.prod(modality_shape_tuple)

                text_tensor = torch.full(
                    (modality_length,), -1, device=device
                )  # text is all -1 here, so text labels are not learned on

                # only add modality meta information when not returning embedding, which only occurs when sampling modality

                succeed_modality_tokens = precede_modality_tokens = 0

                if not return_embed:
                    # add the [som] and [eom] tokens for the modality type

                    som_id, eom_id = mod.som_id, mod.eom_id

                    # start by just storing the token length of the modality

                    modality_shape_str = join([*map(str, modality_shape_tuple)], ",")
                    modality_meta_info = self.char_tokenizer(
                        modality_shape_str, device=device
                    )

                    precede_modality_tokens = len(modality_meta_info) + 2
                    succeed_modality_tokens = 1

                    text_tensor = cat(
                        (
                            tensor_([self.meta_id]),
                            modality_meta_info,
                            tensor_([som_id]),
                            text_tensor,
                            tensor_([eom_id]),
                        )
                    )

                batch_modality_positions.append(
                    (modality_type, offset + precede_modality_tokens, modality_length)
                )  # offset + preceding meta tag length (which includes the modality start token)

                # store parsing out back to shape

                modality_tensor, unpack_modality_shape = pack_one_with_inverse(
                    modality_tensor, "* d"
                )

                inverse_fn = model_to_pred_flow(
                    batch_index,
                    offset + precede_modality_tokens,
                    modality_length,
                    unpack_modality_shape,
                )

                # maybe decorate the function if model output is predicting clean

                if self.model_output_clean:
                    decorator = get_model_output_to_flow_fn(
                        modality_tensor, modality_time, self.eps, return_decorator=True
                    )
                    inverse_fn = decorator(inverse_fn)

                # store function for extracting flow later

                get_pred_flows[modality_type].append(inverse_fn)

                # increment offset

                offset += (
                    modality_length + precede_modality_tokens + succeed_modality_tokens
                )  # +2 due to [som] and [eom] - then account for meta start id and modality shape information (or eventually any meta information about modality)

                modality_tensor = F.pad(
                    modality_tensor,
                    (0, 0, precede_modality_tokens, succeed_modality_tokens),
                )

                batch_modality_tokens.append(modality_tensor)

                batch_text.append(text_tensor)

                # handle axial positional embedding

                if need_axial_pos_emb:

                    if exists(mod.pos_emb_mlp):
                        pos_emb_max_axial_dims[modality_type].append(
                            tensor(modality_shape_tuple)
                        )
                        pos_emb = (
                            modality_type,
                            modality_shape_tuple,
                            (precede_modality_tokens, succeed_modality_tokens),
                        )

                    else:
                        pos_emb = torch.zeros(
                            text_tensor.shape[0], self.dim, device=device
                        )

                    batch_modality_pos_emb.append(pos_emb)

            text.append(cat(batch_text))

            if need_axial_pos_emb:
                modality_pos_emb.append(batch_modality_pos_emb)

            modality_tokens.append(cat(batch_modality_tokens))
            modality_positions.append(batch_modality_positions)

            modality_index += 1

        if return_loss:
            total_tokens = sum([t.numel() for t in text])

        text = pad_sequence(text, padding_value=-1)

        modality_tokens = pad_sequence(modality_tokens, padding_value=0.0)

        # handle modality positional embedding

        if need_axial_pos_emb:
            pos_emb_max_axial_dims = {
                mod_type: stack(sizes, dim=-1).amax(dim=-1)
                for mod_type, sizes in pos_emb_max_axial_dims.items()
            }
            factorized_pos_emb = {
                mod_type: self.get_modality_info(mod_type).pos_emb_mlp(
                    max_size, return_factorized=True
                )
                for mod_type, max_size in pos_emb_max_axial_dims.items()
            }

            # lazy evaluate the modality positional embedding from the factorized positional embedding from maximum axial dims

            evaluated_pos_emb = []

            for batch_modality_pos_emb in modality_pos_emb:
                evaluated_batch_pos_emb = []

                for maybe_pos_emb_config in batch_modality_pos_emb:

                    if is_tensor(maybe_pos_emb_config):
                        evaluated_batch_pos_emb.append(maybe_pos_emb_config)
                        continue

                    mod_type, mod_size, padding = maybe_pos_emb_config

                    mod_info = self.get_modality_info(mod_type)
                    mod_factorized_pos_emb = factorized_pos_emb[mod_type]

                    mod_pos_emb = mod_info.pos_emb_mlp.combine_factorized(
                        mod_factorized_pos_emb, mod_size, flatten=True
                    )
                    mod_pos_emb = F.pad(
                        mod_pos_emb, (0, 0, *padding), value=0.0
                    )  # handle padding for preceding and succeeding meta tokens

                    evaluated_batch_pos_emb.append(mod_pos_emb)

                evaluated_pos_emb.append(cat(evaluated_batch_pos_emb, dim=-2))

            modality_pos_emb = pad_sequence(evaluated_pos_emb, padding_value=0.0)

        # handle training mode and removal of last token

        if return_loss:
            modality_tokens = modality_tokens[:, :-1]

            if need_axial_pos_emb:
                modality_pos_emb = modality_pos_emb[:, :-1]

        # if returning loss, split text for next token prediction

        if return_loss:
            text, text_labels = text[:, :-1], text[:, 1:]

        # derive is_modality mask for flow on the right tokens + flow loss

        batch, seq_len, device = *text.shape, text.device

        assert len(modality_positions) == batch

        if isinstance(modality_positions, list):
            modality_positions = modality_positions_to_tensor(
                modality_positions, device=device
            )

        if (
            modality_positions.shape[-1] == 2
        ):  # Int['b m 2'] -> Int['b m 3'] if type is not given (one modality)
            modality_positions = F.pad(modality_positions, (1, 0))

        # for now use dummy padding modality position info if empty (all zeros)

        if modality_positions.numel() == 0:
            modality_positions = F.pad(modality_positions, (0, 0, 0, 1))

        # sort the modalities tensor and sanitize, readying for noising of modalities

        modality_positions, sorted_indices = order_modality_positions_by_seq_offset(
            modality_positions
        )

        is_modalities = modality_positions_to_is_modality_mask(
            seq_len,
            modality_positions,
            num_modalities=self.num_modalities,
            device=device,
        )

        is_any_modality = reduce(is_modalities, "b t m n -> b n", "any")

        # embed text

        text = text.masked_fill(text == -1, 0)

        text_tokens = self.text_embed(text)

        # maybe add the axial positional embedding

        if need_axial_pos_emb:
            modality_tokens = modality_tokens + modality_pos_emb

        # intersperse the modalities with the text for the joint transformer + flow system

        tokens = einx.where(
            "b n, b n d, b n d", is_any_modality, modality_tokens, text_tokens
        )

        # derive rotary positions

        rotary_positions = derive_rotary_positions_from_modality_positions(
            seq_len, modality_positions
        )

        rotary_emb = self.rotary_emb(rotary_positions)
        rotary_emb = rearrange(rotary_emb, "b n d -> b 1 n d")

        # take care of cache

        is_any_modality_when_decoding = None

        if exists(cache):
            assert exists(
                decode_length
            ), "`decode_length` must be passed in on forward for modality sampling. think of a cleaner way on some future date"
            assert exists(decoding_text_or_modality)

            if decoding_text_or_modality == "text":
                decode_length = 1

            is_any_modality_when_decoding = decoding_text_or_modality == "modality"
            modality_positions = None

        # times

        times_per_token = einsum(is_modalities.float(), times, "b t m n, b m -> b t n")

        times_cond = reduce(times_per_token, "b t n -> b n", "sum")

        # attention

        embed, hiddens, *maybe_kv_cache = self.transformer(
            tokens,
            times=times_cond,
            rotary_emb=rotary_emb,
            modality_positions=modality_positions,
            is_any_modality=is_any_modality_when_decoding,
            cache=cache,
            decode_length=decode_length,
            return_hiddens=True,
            return_kv_cache=return_kv_cache,
        )

        kv_cache = maybe_kv_cache[0] if return_kv_cache else None

        # helper for appending auxiliary returns

        def maybe_pack_aux(out):
            ret = (out,)

            if return_kv_cache:
                ret = (*ret, kv_cache)

            if return_hiddens:
                ret = (*ret, hiddens)

            if return_times:
                ret = (*ret, times)

            if len(ret) == 1:
                return ret[0]

            return ret

        # early return for embedding for decoding modality

        if return_embed:
            return maybe_pack_aux((embed, get_pred_flows))

        # text unembedding

        text_logits = self.to_text_logits(embed)

        if not return_loss:
            return maybe_pack_aux(text_logits)

        # flow loss

        pred_flows = []
        recon_losses = []

        for modality_id in range(self.num_modalities):
            mod = self.get_modality_info(modality_id)

            modality_get_pred_flows = get_pred_flows[modality_id]
            modality_get_recon_losses = get_recon_losses[modality_id]

            modality_pred_flows = []
            modality_recon_losses = []

            for get_pred_flow, get_recon_loss in zip(
                modality_get_pred_flows, modality_get_recon_losses
            ):

                pred_flow = get_pred_flow(embed)
                pred_flow = add_temp_batch_dim(mod.model_to_latent)(pred_flow)
                modality_pred_flows.append(pred_flow)

                if not return_loss or not self.has_recon_loss:
                    continue

                modality_recon_losses.append(get_recon_loss(pred_flow))

            pred_flows.append(modality_pred_flows)
            recon_losses.append(modality_recon_losses)

        # early return for velocity consistency ema model

        if return_only_pred_flows:
            return pred_flows

        # text autoregressive loss

        text_labels = text_labels.masked_fill(is_any_modality, self.ignore_index)

        # ignore "Image -> Null" mappings.
        text_labels = text_labels.masked_fill(
            text_labels == self.null_text_id, self.ignore_index
        )

        text_loss = F.cross_entropy(
            rearrange(text_logits, "b n l -> b l n"),
            text_labels,
            ignore_index=self.ignore_index,
        )

        text_loss_weight = (text_labels != self.ignore_index).sum() / total_tokens

        # calculate flow losses

        flow_losses = []

        modality_loss_weights = []

        for modality_id, (pred_flow, is_one_modality) in enumerate(
            zip(pred_flows, is_modalities.unbind(dim=1))
        ):
            mod = self.get_modality_info(modality_id)

            is_one_modality = reduce(is_one_modality, "b m n -> b n", "any")
            modality_loss_weight = is_one_modality.sum() / total_tokens

            modality_flows = flows[modality_id]

            pack_pattern = "d *" if mod.channel_first_latent else "* d"

            modality_pred_flow, _ = pack(pred_flow, pack_pattern)
            modality_flows, _ = pack(modality_flows, pack_pattern)

            flow_loss = F.mse_loss(modality_pred_flow, modality_flows)

            modality_loss_weights.append(modality_loss_weight)

            flow_losses.append(flow_loss)

        modality_loss_weights = stack(modality_loss_weights)

        # only the token positions that are not modalities have autoregressive loss

        total_loss = (
            text_loss * text_loss_weight * self.text_loss_weight
            + (stack(flow_losses) * modality_loss_weights).sum() * self.flow_loss_weight
        )

        # whether to handle velocity consistency
        # for straightening the flow, from consistency flow matching paper https://arxiv.org/abs/2407.02398

        velocity_match_losses = None

        if need_velocity_matching:

            with torch.no_grad():
                velocity_consistency_ema_model.eval()

                ema_pred_flows = velocity_consistency_ema_model(
                    velocity_modalities,
                    times=orig_times + velocity_consistency_delta_time,
                    return_only_pred_flows=True,
                )

            velocity_match_losses = []

            for ema_pred_flow, pred_flow in zip(ema_pred_flows, pred_flows):

                pack_pattern = "d *" if mod.channel_first_latent else "* d"
                pred_flow, _ = pack(pred_flow, pack_pattern)
                ema_pred_flow, _ = pack(ema_pred_flow, pack_pattern)

                velocity_match_loss = F.mse_loss(pred_flow, ema_pred_flow)

                velocity_match_losses.append(velocity_match_loss)

            total_loss = (
                total_loss
                + (stack(velocity_match_losses) * modality_loss_weights).sum()
                * self.velocity_consistency_loss_weight
            )

        # maybe reconstruction loss

        if self.has_recon_loss:

            averaged_recon_losses = []

            for modality_recon_loss in recon_losses:
                averaged_recon_losses.append(
                    sum(modality_recon_loss) / len(modality_recon_loss)
                )

            total_loss = (
                total_loss
                + (stack(averaged_recon_losses) * modality_loss_weights).sum()
                * self.reconstruction_loss_weight
            )

        # return total loss and maybe breakdown

        if not return_breakdown and not return_hiddens and not return_times:
            return total_loss

        ret = (total_loss,)

        if return_breakdown:
            breakdown = LossBreakdown(
                total_loss, text_loss, flow_losses, velocity_match_losses, recon_losses
            )
            ret = (*ret, breakdown)

        if return_hiddens:
            ret = (*ret, hiddens)

        if return_times:
            ret = (*ret, times)

        return ret

#### Run Training loop

In [36]:
from accelerate import Accelerator

# constants

IMAGE_AFTER_TEXT = True   # False for captioning, True for text-to-image
USE_PROMPT = False        # whether to use prompting, or synthesize from start token
NUM_TRAIN_STEPS = 20_000
SAMPLE_EVERY = 500
CHANNEL_FIRST = True


In [37]:
# functions

def divisible_by(num, den):
    return (num % den) == 0

# encoder / decoder

class Encoder(Module):
    def forward(self, x):
        x = rearrange(x, '... 1 (h p1) (w p2) -> ... h w (p1 p2)', p1 = 2, p2 = 2)

        if CHANNEL_FIRST:
            x = rearrange(x, 'b ... d -> b d ...')

        return x * 2 - 1

class Decoder(Module):
    def forward(self, x):

        if CHANNEL_FIRST:
            x = rearrange(x, 'b d ... -> b ... d')

        x = rearrange(x, '... h w (p1 p2) -> ... 1 (h p1) (w p2)', p1 = 2, p2 = 2)
        return ((x + 1) * 0.5).clamp(min = 0., max = 1.)

In [38]:
model = Transfusion(
    num_text_tokens = 10,
    dim_latent = 4,
    modality_default_shape = (14, 14),
    modality_encoder = Encoder(),
    modality_decoder = Decoder(),
    add_pos_emb = True,
    modality_num_dim = 2,
    prob_uncond = 0.1,
    channel_first_latent = CHANNEL_FIRST,
    transformer = dict(
        dim = 64,
        depth = 4,
        dim_head = 32,
        heads = 8,
    )
)

ema_model = model.create_ema()

In [39]:
class MnistDataset(Dataset):
    def __init__(self):
        self.mnist = torchvision.datasets.MNIST(
            DATA_DIRECTORY,
            download = True
        )

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        pil, labels = self.mnist[idx]
        digit_tensor = T.PILToTensor()(pil)
        output =  tensor(labels), (digit_tensor / 255).float()

        if IMAGE_AFTER_TEXT:
            return output

        first, second = output
        return second, first
    
def cycle(iter_dl):
    while True:
        for batch in iter_dl:
            yield batch

def collate_fn(data):
    data = [*map(list, data)]
    return data

In [40]:
import os

results_path = os.path.join(RESULTS_DIRECTORY, "transfusion", "multimodal")
results_folder = Path(results_path)

rmtree(results_path, ignore_errors=True)
results_folder.mkdir(exist_ok=True, parents=True)

In [41]:
dataset = MnistDataset()
dataloader = model.create_dataloader(dataset, batch_size = 16, shuffle = True)

iter_dl = cycle(dataloader)

In [42]:
optimizer = Adam(model.parameters(), lr = 3e-4)

accelerator = Accelerator()

model, optimizer, dataloader = accelerator.prepare(model, optimizer, dataloader)

ema_model.to(accelerator.device)


EMA(
  (online_model): Transfusion(
    (transformer): Transformer(
      (to_time_cond): Sequential(
        (0): RandomFourierEmbed()
        (1): Linear(in_features=65, out_features=256, bias=True)
        (2): SiLU()
      )
      (expand_stream): Reduce('... d -> ... s d', 'repeat', s=1)
      (reduce_stream): Reduce('... s d -> ... d', 'sum')
      (layers): ModuleList(
        (0): ModuleList(
          (0): None
          (1): AdaptiveWrapper(
            (fn): Attention(
              (to_qk): Sequential(
                (0): Linear(in_features=64, out_features=512, bias=False)
                (1): Rearrange('b n (qk h d) -> qk b h n d', qk=2, h=8)
              )
              (to_v): Sequential(
                (0): Linear(in_features=64, out_features=256, bias=False)
                (1): Rearrange('b n (h d) -> b h n d', h=8)
              )
              (to_gates): Sequential(
                (0): Linear(in_features=64, out_features=8, bias=False)
                (1): Rea

In [43]:
# train loop

for step in range(1, NUM_TRAIN_STEPS + 1):
    model.train()

    loss = model(next(iter_dl))
    accelerator.backward(loss)

    accelerator.clip_grad_norm_(model.parameters(), 0.5)

    optimizer.step()
    optimizer.zero_grad()

    ema_model.update()

    accelerator.print(f'{step}: {loss.item():.3f}')

    # eval

    if divisible_by(step, SAMPLE_EVERY):

        GUIDANCE_SCALE = 3.0

        if not USE_PROMPT:
            # sampling from start to finish

            one_multimodal_sample = ema_model.sample(max_length = 384, cfg_scale = GUIDANCE_SCALE)

        else:
            # sampling using prompt
            # which differs depending on which comes first, text or images

            if IMAGE_AFTER_TEXT:

                text_label = torch.randint(0, 10, ()).to(accelerator.device)
                one_multimodal_sample = ema_model.sample(prompt = text_label, max_length = 384, cfg_scale = GUIDANCE_SCALE)

            else:

                rand_batch = next(iter_dl)
                rand_image = rand_batch[0][0]

                one_multimodal_sample = ema_model.sample(prompt = rand_image, max_length = 384, cfg_scale = GUIDANCE_SCALE)

        # make sure modality sample overall order of modalities look correct

        print_modality_sample(one_multimodal_sample)

        if len(one_multimodal_sample) < 2:
            continue

        if IMAGE_AFTER_TEXT:
            maybe_label, maybe_image, *_ = one_multimodal_sample
        else:
            _, maybe_image, maybe_label = one_multimodal_sample

        filename = f'{step}.{maybe_label[1].item()}.png'

        if accelerator.is_main_process:
            save_image(
                maybe_image[1].cpu(),
                str(results_folder / filename),
            )


1: 218.603
2: 70.743
3: 9.115
4: 4.030
5: 2.379
6: 15.642
7: 1.771
8: 2.073
9: 1.963
10: 7.031
11: 3.505
12: 1.876
13: 1.752
14: 1.782
15: 3.337
16: 1.766
17: 1.772
18: 3.440
19: 9.109
20: 1.982
21: 2.807
22: 1.566
23: 5.934
24: 1.533
25: 5.817
26: 1.473
27: 4.285
28: 1.604
29: 1.560
30: 2.962
31: 2.294
32: 1.412
33: 1.529
34: 1.676
35: 1.404
36: 2.263
37: 1.422
38: 1.303
39: 1.321
40: 3.494
41: 1.391
42: 3.076
43: 1.319
44: 1.197
45: 1.237
46: 3.695
47: 1.277
48: 1.194
49: 2.116
50: 3.004
51: 2.020
52: 1.287
53: 1.194
54: 1.412
55: 2.187
56: 1.049
57: 1.426
58: 1.102
59: 1.291
60: 1.321
61: 1.448
62: 3.327
63: 0.989
64: 1.109
65: 0.956
66: 0.932
67: 1.235
68: 0.949
69: 0.959
70: 1.061
71: 1.063
72: 1.792
73: 0.954
74: 0.836
75: 0.779
76: 0.848
77: 0.839
78: 0.946
79: 0.814
80: 2.920
81: 0.824
82: 0.752
83: 0.747
84: 0.692
85: 0.682
86: 0.678
87: 2.366
88: 0.842
89: 0.720
90: 1.297
91: 0.862
92: 1.255
93: 1.062
94: 1.024
95: 0.794
96: 0.743
97: 0.666
98: 0.926
99: 0.684
100: 1.338
101:

decoding text:  53%|█████▎    | 205/384 [00:11<00:10, 17.77it/s]
2026-04-19 13:48:11.737 | INFO     | __main__:sample:707 - sampling stopped at length: 205 / 384
2026-04-19 13:48:11.738 | INFO     | transfusion_pytorch.transfusion:print_modality_sample:261 - [('text', torch.Size([9])), ('modality:0', torch.Size([1, 28, 28])), ('text', torch.Size([2]))]


501: 0.350
502: 0.378
503: 0.440
504: 0.359
505: 0.413
506: 0.357
507: 0.401
508: 0.566
509: 0.474
510: 0.396
511: 0.414
512: 0.555
513: 0.519
514: 0.338
515: 0.411
516: 0.406
517: 0.388
518: 0.390
519: 0.444
520: 0.388
521: 0.353
522: 0.424
523: 0.365
524: 0.308
525: 0.407
526: 0.385
527: 0.380
528: 0.435
529: 0.479
530: 0.450
531: 0.364
532: 0.397
533: 0.386
534: 0.383
535: 0.623
536: 0.514
537: 0.311
538: 0.407
539: 0.373
540: 0.376
541: 0.442
542: 0.328
543: 0.503
544: 0.516
545: 0.343
546: 0.524
547: 0.365
548: 0.334
549: 0.361
550: 0.360
551: 1.018
552: 0.382
553: 0.638
554: 0.385
555: 0.387
556: 0.670
557: 0.393
558: 0.564
559: 0.412
560: 0.404
561: 0.371
562: 0.329
563: 0.548
564: 0.421
565: 0.411
566: 0.384
567: 0.462
568: 0.409
569: 0.426
570: 0.434
571: 0.831
572: 0.424
573: 0.400
574: 0.412
575: 0.354
576: 0.396
577: 0.624
578: 0.394
579: 0.434
580: 0.386
581: 0.424
582: 0.407
583: 0.357
584: 0.584
585: 0.355
586: 0.471
587: 0.368
588: 0.455
589: 0.416
590: 0.418
591: 0.688

decoding text:  53%|█████▎    | 205/384 [00:01<00:01, 120.18it/s]
2026-04-19 13:49:34.649 | INFO     | __main__:sample:707 - sampling stopped at length: 205 / 384
2026-04-19 13:49:34.650 | INFO     | transfusion_pytorch.transfusion:print_modality_sample:261 - [('text', torch.Size([9])), ('modality:0', torch.Size([1, 28, 28])), ('text', torch.Size([2]))]


1001: 0.360
1002: 0.306
1003: 0.387
1004: 0.378
1005: 0.565
1006: 0.371
1007: 0.407
1008: 0.381
1009: 0.504
1010: 0.343
1011: 0.350
1012: 0.554
1013: 0.349
1014: 0.303
1015: 0.344
1016: 0.351
1017: 0.333
1018: 0.330
1019: 0.414
1020: 0.314
1021: 0.295
1022: 0.475
1023: 0.373
1024: 0.380
1025: 0.353
1026: 0.329
1027: 0.467
1028: 0.411
1029: 0.345
1030: 1.003
1031: 0.835
1032: 0.406
1033: 0.376
1034: 0.336
1035: 0.496
1036: 0.391
1037: 0.450
1038: 0.466
1039: 0.359
1040: 0.459
1041: 0.377
1042: 0.368
1043: 0.346
1044: 0.450
1045: 0.537
1046: 0.373
1047: 0.445
1048: 0.368
1049: 0.360
1050: 0.550
1051: 0.360
1052: 0.349
1053: 0.389
1054: 0.351
1055: 0.599
1056: 0.332
1057: 0.336
1058: 0.348
1059: 0.340
1060: 0.353
1061: 0.347
1062: 0.512
1063: 0.317
1064: 0.327
1065: 0.335
1066: 0.356
1067: 0.332
1068: 0.416
1069: 0.357
1070: 0.471
1071: 0.524
1072: 0.374
1073: 0.358
1074: 0.340
1075: 0.360
1076: 0.393
1077: 0.260
1078: 0.339
1079: 0.317
1080: 0.372
1081: 0.368
1082: 0.373
1083: 0.545
1084

KeyboardInterrupt: 

In [13]:
loss

tensor(343.3325, device='mps:0', grad_fn=<AddBackward0>)